# 你的第一个振动信号

## 学习目标

读完本页并跑完代码，你应该能够：

1. 说出“采样率”和“奈奎斯特频率”分别是什么，并算出本例的奈奎斯特频率。
2. 用 NumPy 生成一段带噪声的模拟振动信号，并画出时域波形。
3. 用 FFT 画出频谱，并指出信号里包含哪些频率成分。

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JingeW/AI_turorial/blob/main/book/module-0-orientation/first-signal.ipynb)

## 问题：传感器给了你一串数字，里面有什么？

假设你在一台旋转机械上装了加速度计，得到一长串按固定时间间隔采样的数字。
肉眼看波形，只能看到“抖动”。如果轴承出了问题，故障信息往往藏在**频率**里，而不是波形的形状里。
后面几章我们会用真实轴承数据做故障分类，这一页先用模拟信号熟悉最基本的工具。

## 概念：时域和频域

- **时域**：信号随时间变化的曲线，横轴是时间。
- **频域**：把信号分解成不同频率的正弦波，横轴是频率，纵轴是各频率成分的强度。
- **FFT**（快速傅里叶变换）：从时域算出频域的标准算法。
- **采样率** $f_s$：每秒采样多少次。能被正确识别的最高频率是 $f_s/2$，叫奈奎斯特频率。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=0)

fs = 2000          # sampling rate in Hz
duration = 1.0     # seconds
t = np.arange(0, duration, 1 / fs)

# Two sinusoids (50 Hz and 120 Hz) plus Gaussian noise
signal = (
    1.0 * np.sin(2 * np.pi * 50 * t)
    + 0.5 * np.sin(2 * np.pi * 120 * t)
    + 0.3 * rng.standard_normal(t.size)
)

print(f"samples: {signal.size}, Nyquist frequency: {fs / 2:.0f} Hz")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(t[:400], signal[:400])
ax.set_xlabel("Time (s)")
ax.set_ylabel("Acceleration (a.u.)")
ax.set_title("Time domain (first 0.2 s)")
plt.show()

In [ ]:
spectrum = np.abs(np.fft.rfft(signal)) / signal.size * 2
freqs = np.fft.rfftfreq(signal.size, d=1 / fs)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(freqs, spectrum)
ax.set_xlim(0, 300)
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Amplitude (a.u.)")
ax.set_title("Frequency domain")
plt.show()

## 练习

把第二个正弦波的频率从 120 Hz 改成 1500 Hz，重新运行。频谱里峰出现在哪个频率？为什么？

<details>
<summary>参考答案</summary>

峰会出现在 500 Hz，而不是 1500 Hz。本例采样率是 2000 Hz，奈奎斯特频率是 1000 Hz，
1500 Hz 超过了它，会被“折叠”到 $2000 - 1500 = 500$ Hz。这种现象叫**混叠**。

</details>

## 常见陷阱

- 采样率不够高：信号里高于奈奎斯特频率的成分不会消失，而是以错误的频率出现在频谱里（混叠）。
- 只看波形下结论：很多故障特征在波形上看不出来，需要看频谱。
- 忽略单位和幅值缩放：不同软件的 FFT 归一化约定不同，比较结果前先确认。

## 延伸阅读

- NumPy 文档：`numpy.fft` 模块。
- 任意一本信号处理入门教材中关于采样定理的章节。